# 02 — Pipeline & Inferenz

Phase 3 (Pipeline bauen + erste Inferenz auf 12 Hand-Gold-Anzeigen), Phase 4 (Iterationen A + B — pro Iteration eigener Predictions-Dateiname und Run-Header-Update), Phase 6 (voller Korpus auf 7B + 3B-Kontrast auf euler).

Cheatsheets: `CHEATSHEETS/transformers-konzepte.md` (Modell, Chat-Template, JSON-Parsing), `CHEATSHEETS/gpu-zugang.md` (Spawn, GPU-Wahl, Memory).

## Run-Header

| Feld | Wert |
|---|---|
| Datum | _YYYY-MM-DD_ |
| Modell | _ (z. B. `Qwen/Qwen2.5-7B-Instruct`) |
| Server | _ (gauss / euler) |
| GPU-Index | _ |
| Schema-Datei | `SCHEMA.md` |
| Aktueller Run-Tag | _ (`baseline` / `iter_A` / `iter_B` / `full_7b` / `full_3b`) |
| Predictions-Datei | _ (`predictions.jsonl` / `predictions_iter_A.jsonl` / …) |
| Truncation | _ Zeichen (initial ~2000) |

Bei jeder neuen Iteration: Run-Tag + Predictions-Datei + Datum aktualisieren.

## Phase 3 — Pipeline bauen + Baseline-Inferenz

### Block 3.1 — Modell laden + Pipeline-Skelett

Konventionen aus `CHEATSHEETS/transformers-konzepte.md`:
- `Qwen/Qwen2.5-7B-Instruct`, `torch_dtype=torch.float32`, `.to("cuda").eval()` (kein `device_map="auto"`)
- Chat-Template via `tokenizer.apply_chat_template(...)`
- `do_sample=False`, `max_new_tokens=200`, `pad_token_id=tokenizer.eos_token_id`
- Robustes JSON-Parsen mit Regex statt direktem `json.loads`
- Head-Truncation auf ~2000 Zeichen (V100/T100, 32 GB)

Predictions landen in `predictions.jsonl` (git-ignored). Eine Zeile pro Anzeige: `{"refnr": ..., <6 Schema-Felder>}`.

In [ ]:
# WICHTIG: CUDA_VISIBLE_DEVICES MUSS vor `import torch` gesetzt werden, sonst greift es nicht.
# GPU 0 ist auf gauss oft von einem anderen Prozess belegt — wir nehmen GPU 1.
# Nach Setzen sieht PyTorch nur diese eine GPU, intern als `cuda:0` adressiert.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import json
import re
import time
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME       = "Qwen/Qwen2.5-7B-Instruct"
KORPUS_PATH      = Path("../daten/eigener_korpus.jsonl")
GOLD_PATH        = Path("../annotation/meine_gold.csv")
PRED_PATH        = Path("predictions.jsonl")          # baseline
MAX_INPUT_CHARS  = 2000
MAX_NEW_TOKENS   = 200

print(f"CUDA_VISIBLE_DEVICES = {os.environ.get('CUDA_VISIBLE_DEVICES')}")
print(f"sichtbare GPUs       : {torch.cuda.device_count()}")
print(f"aktive GPU           : {torch.cuda.current_device()} = {torch.cuda.get_device_name(0)}")
free, total = torch.cuda.mem_get_info(0)
print(f"GPU-Speicher frei    : {free / 1e9:.1f} GB / {total / 1e9:.1f} GB total")

In [ ]:
# Modell + Tokenizer laden (V100/T100-Constraints: float32, kein device_map)
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
).to("cuda").eval()
print(f"Modell {MODEL_NAME} geladen in {time.time() - t0:.1f}s")
print(f"GPU-Speicher belegt: {torch.cuda.memory_allocated() / 1e9:.1f} GB / "
      f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# System-Prompt = Schema + ein konkretes Beispiel-JSON (stabilisiert Qwen-Output stark)

SYSTEM_PROMPT = """Du extrahierst strukturierte Informationen aus deutschen Stellenanzeigen.
Antworte AUSSCHLIESSLICH mit einem gültigen JSON-Objekt nach folgendem Schema — kein Begleittext, keine Markdown-Codefences.

Schema (alle 6 Felder zwingend vorhanden):
- "homeoffice": "ja" | "teilweise" | "nein" | "remote" | "nicht_genannt"
- "vertragsart": "ausbildung" | "festanstellung" | "praktikum" | "werkstudent" | "sonstiges"
- "erfahrungslevel": "junior" | "mid" | "senior" | "egal" | "nicht_genannt"
- "gehalt_min_eur": ganze Zahl (Untergrenze) ODER null
- "gehalt_zeitraum": "monat" | "jahr" | null  (null nur, wenn gehalt_min_eur null ist)
- "skills_top3": Array mit max. 3 technischen Skills aus dem Text (z. B. "Python", "SQL", "Power BI") — leeres Array, wenn keine genannt

Regeln:
- "nicht_genannt" NUR, wenn die Anzeige zum Feld wirklich nichts sagt — nicht als Sicherheits-Antwort bei Unsicherheit.
- Bei einer Range ("ab 50.000 €" / "50.000–60.000 €") nimm die untere Grenze als ganze Zahl ohne Tausender-Trennung.
- Ausbildungen sind immer "junior". Werkstudent/Praktikum → "praktikum"/"werkstudent" + erfahrungslevel meist "junior".
- skills_top3: nur konkrete Tools/Sprachen/Frameworks — KEINE Soft Skills ("Teamfähigkeit"), KEINE Sprachen ("Englisch"), KEINE Fachgebiete ("Informatik").

Beispiel-Output:
{"homeoffice": "teilweise", "vertragsart": "festanstellung", "erfahrungslevel": "mid", "gehalt_min_eur": 55000, "gehalt_zeitraum": "jahr", "skills_top3": ["Python", "SQL", "Power BI"]}"""


def baue_messages(text: str) -> list[dict]:
    """System + User-Turn. Anzeige wird auf MAX_INPUT_CHARS Zeichen head-truncated."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Stellenanzeige:\n\n{text[:MAX_INPUT_CHARS]}"},
    ]


def extrahiere_json(response: str) -> dict | None:
    """Zieht das erste JSON-Objekt aus dem Modell-Output. Returns None bei Parse-Fail."""
    match = re.search(r"\{.*\}", response, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None


def inferenz(text: str) -> tuple[dict | None, str]:
    """Pro Anzeige: prompt → generate → JSON-Parse. Returns (parsed_dict_or_None, raw_response)."""
    messages = baue_messages(text)
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True,
    )
    return extrahiere_json(response), response

### Block 3.2 — Inferenz auf 12 Hand-Gold-Anzeigen

Wir filtern den Korpus auf die 12 IDs aus `meine_gold.csv` und schreiben pro Anzeige eine JSONL-Zeile. Erwartete Laufzeit auf V100/T100 (7B fp32): ~40–60 s.

In [ ]:
# 12 Hand-Gold-IDs aus meine_gold.csv laden und Korpus filtern
gold_df = pd.read_csv(GOLD_PATH)
gold_ids = gold_df["id"].astype(str).tolist()

korpus = pd.read_json(KORPUS_PATH, lines=True)
anzeigen = korpus[korpus["refnr"].isin(gold_ids)].copy()
anzeigen = anzeigen.set_index("refnr").loc[gold_ids].reset_index()   # gleiche Reihenfolge wie gold

print(f"Hand-Gold-IDs : {len(gold_ids)}")
print(f"im Korpus     : {len(anzeigen)}")
assert len(anzeigen) == len(gold_ids), "Fehlende refnr — meine_gold.csv vs. eigener_korpus.jsonl vergleichen"

In [ ]:
# Inferenz-Schleife → predictions.jsonl
predictions  = []
parse_fails  = 0
t_start      = time.time()

with PRED_PATH.open("w", encoding="utf-8") as f:
    for i, row in enumerate(anzeigen.itertuples(index=False), 1):
        t_i = time.time()
        parsed, raw = inferenz(row.text)
        dt = time.time() - t_i

        if parsed is None:
            parse_fails += 1
            entry = {"refnr": row.refnr, "_parse_fail": True, "_raw": raw[:400]}
            tag = "PARSE_FAIL"
        else:
            entry = {"refnr": row.refnr, **parsed}
            tag = "OK"

        f.write(json.dumps(entry, ensure_ascii=False) + "\n")
        predictions.append(entry)
        print(f"  [{i:02d}/{len(anzeigen)}] {row.refnr}  {tag:<10}  ({dt:.1f}s)")

print(f"\nFertig: {len(predictions)} Anzeigen in {time.time() - t_start:.0f}s")
print(f"JSON-Parse-Fails: {parse_fails}/{len(predictions)}")
print(f"Predictions → {PRED_PATH.resolve()}")

## Phase 4 — Iteration A: Prompt-Klarstellung (Homeoffice)

**Hypothese:** Schwächstes Feld in der Baseline ist `homeoffice` (κ zwischen Menschen war schon nur 0.122 — Schema-Ambivalenz zwischen `ja`/`teilweise`/`nicht_genannt`).
**Hebel A:** Prompt mit präzisen Regeln pro Homeoffice-Wert + explizitem Edge-Case-Mapping aus den Phase-2-Befunden.
**Erwartung:** +2–4 korrekte `homeoffice`-Klassifikationen (entspricht +17–34 Pt). Wenn das hilft, ist's ein **Schema-Problem** (Definitions-Lücke), kein reines Modell-Problem.
**Predictions:** `predictions_iter_A.jsonl`. Run-Header oben aktualisieren.

In [ ]:
# Schärferer System-Prompt mit präzisen Homeoffice-Regeln + Edge-Case-Mapping
SYSTEM_PROMPT_A = """Du extrahierst strukturierte Informationen aus deutschen Stellenanzeigen.
Antworte AUSSCHLIESSLICH mit einem gültigen JSON-Objekt — kein Begleittext, keine Markdown-Codefences.

Schema (alle 6 Felder zwingend vorhanden):
- "homeoffice": "ja" | "teilweise" | "nein" | "remote" | "nicht_genannt"
- "vertragsart": "ausbildung" | "festanstellung" | "praktikum" | "werkstudent" | "sonstiges"
- "erfahrungslevel": "junior" | "mid" | "senior" | "egal" | "nicht_genannt"
- "gehalt_min_eur": ganze Zahl (Untergrenze) ODER null
- "gehalt_zeitraum": "monat" | "jahr" | null
- "skills_top3": Array mit max. 3 technischen Skills

PRÄZISE HOMEOFFICE-REGELN (Hauptproblem in der Baseline):
- "remote"        → NUR bei "100% Home Office", "Vollzeit Remote", "ortsunabhängig", "deutschlandweit von zu Hause"
- "teilweise"     → wenn "hybrid", "X Tage Homeoffice", "anteilig", "mobiles Arbeiten", "Gleitzeit und Homeoffice" EXPLIZIT
- "ja"            → wenn "Homeoffice" / "Home Office" steht, OHNE Modalität (weder klar remote noch eindeutig hybrid)
- "nein"          → "Präsenzpflicht", "vor Ort zwingend", "kein Homeoffice"
- "nicht_genannt" → die Anzeige sagt NICHTS dazu (auch nicht im Benefits-Block)

Edge-Cases (aus echten Anzeigen):
- "Möglichkeit zum hybriden Arbeiten"   → teilweise  (Wort "hybrid" reicht)
- "Homeoffice nach Absprache"           → teilweise  (impliziert Hybrid-Modell)
- "Gleitzeitregelung und Homeoffice"    → teilweise  (Kombination → Hybrid)

VERTRAGSART-REGELN:
- "Ausbildung" / "Azubi"                → ausbildung; erfahrungslevel IMMER junior
- "Werkstudent"                         → werkstudent
- "Praktikum" / "Praktikant"            → praktikum
- "Freelance" / "Lehrbeauftragter"      → sonstiges
- sonst regulärer Vertrag               → festanstellung

GEHALT-REGELN:
- Range "ab 50.000 €" / "50.000–60.000 €" → 50000 als Untergrenze (Zahl ohne Tausender-Trennung)
- "nach Vereinbarung" / "attraktive Vergütung" / "marktüblich" → null
- Wenn gehalt_min_eur = null → gehalt_zeitraum MUSS null sein

SKILLS-REGELN:
- KEINE Soft Skills ("Teamfähigkeit", "Kommunikationsfähigkeit")
- KEINE Sprachen ("Deutsch", "Englisch")
- KEINE Fachgebiete ("Informatik", "BWL")
- NUR konkrete Tools/Sprachen/Frameworks/Methoden: Python, SQL, Power BI, AWS, Machine Learning, ETL, …
- Maximal 3, leeres Array wenn keine technischen Skills genannt

Beispiel-Output:
{"homeoffice": "teilweise", "vertragsart": "festanstellung", "erfahrungslevel": "mid", "gehalt_min_eur": 55000, "gehalt_zeitraum": "jahr", "skills_top3": ["Python", "SQL", "Power BI"]}"""


def baue_messages_A(text: str) -> list[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT_A},
        {"role": "user",   "content": f"Stellenanzeige:\n\n{text[:MAX_INPUT_CHARS]}"},
    ]


def inferenz_A(text: str) -> tuple[dict | None, str]:
    messages = baue_messages_A(text)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return extrahiere_json(response), response


# Run Iteration A auf den 12 Hand-Gold-Anzeigen
PRED_PATH_A = Path("predictions_iter_A.jsonl")
predictions_A, parse_fails_A = [], 0
t_start = time.time()

with PRED_PATH_A.open("w", encoding="utf-8") as f:
    for i, row in enumerate(anzeigen.itertuples(index=False), 1):
        t_i = time.time()
        parsed, raw = inferenz_A(row.text)
        dt = time.time() - t_i
        if parsed is None:
            parse_fails_A += 1
            entry, tag = {"refnr": row.refnr, "_parse_fail": True, "_raw": raw[:400]}, "PARSE_FAIL"
        else:
            entry, tag = {"refnr": row.refnr, **parsed}, "OK"
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")
        predictions_A.append(entry)
        print(f"  [{i:02d}/{len(anzeigen)}] {row.refnr}  {tag:<10}  ({dt:.1f}s)")

print(f"\nIteration A fertig: {len(predictions_A)} in {time.time()-t_start:.0f}s, {parse_fails_A} Parse-Fails")
print(f"Predictions → {PRED_PATH_A.resolve()}")

## Phase 4 — Iteration B: Few-Shot-Beispiele

**Hypothese:** Selbst mit präzisem Schema-Prompt verfehlt das Modell manche Werte, weil das Format nur abstrakt beschrieben ist. **Konkrete Vorbild-JSONs** in der Message-History stabilisieren Format und Werte-Konsistenz stärker als Schema-Text allein.
**Hebel B:** 3 Few-Shot-Paare `user`/`assistant` aus dem Hand-Gold *vor* dem aktuellen User-Turn — eine Auswahl, die das Spektrum abdeckt (Ausbildung, Senior-Festanstellung, Werkstudent).
**Erwartung:** Vor allem `skills_top3` (Format korrekter, weniger Soft Skills) und `gehalt_min_eur`/`gehalt_zeitraum`-Konsistenz besser. `homeoffice` wahrscheinlich kaum, weil die Beispiele die Ambivalenz nicht auflösen.
**Predictions:** `predictions_iter_B.jsonl`.

> **Caveat:** Few-Shots kommen aus den 12 Test-IDs → leichte Test-Contamination. Workshop-Mechanik im Vordergrund; in Production würde man die Few-Shots aus einem separaten Holdout ziehen.

In [ ]:
# Drei Few-Shot-Beispiele aus dem Hand-Gold (diverse Mix: Ausbildung, Senior-Festanstellung, Werkstudent)
FEW_SHOT_IDS = [
    "10000-1203863577-S",      # Ausbildung → junior
    "11949-17214039-S",        # Festanstellung, senior
    "13151-1568687-1-S",       # Werkstudent
]

def gold_zu_json(row: dict) -> dict:
    """meine_gold.csv-Zeile → JSON-Dict im Pipeline-Format (mit null für leer)."""
    def _str(v):
        if v is None or (isinstance(v, float) and pd.isna(v)):
            return None
        s = str(v).strip()
        return None if s.lower() in ("", "null", "nan") else s
    def _int(v):
        s = _str(v)
        if s is None: return None
        try: return int(float(s))
        except (ValueError, TypeError): return None
    def _skills(v):
        s = _str(v)
        if s is None or s.lower() == "nicht_genannt":
            return []
        return [x.strip() for x in s.split("|") if x.strip() and x.strip().lower() != "nicht_genannt"][:3]
    return {
        "homeoffice":      _str(row.get("homeoffice")),
        "vertragsart":     _str(row.get("vertragsart")),
        "erfahrungslevel": _str(row.get("erfahrungslevel")),
        "gehalt_min_eur":  _int(row.get("gehalt_min_eur")),
        "gehalt_zeitraum": _str(row.get("gehalt_zeitraum")),
        "skills_top3":     _skills(row.get("skills_top3")),
    }

gold_lookup   = {str(r["id"]): dict(r) for _, r in gold_df.iterrows()}
korpus_lookup = {str(r["refnr"]): dict(r) for _, r in korpus.iterrows()}

few_shots = []
MAX_INPUT_CHARS_B = 1200      # enger als Baseline, weil 3 Few-Shots dazukommen
for fs_id in FEW_SHOT_IDS:
    fs_text = korpus_lookup[fs_id]["text"][:MAX_INPUT_CHARS_B]
    fs_gold = gold_zu_json(gold_lookup[fs_id])
    few_shots.append((fs_text, fs_gold))
    print(f"  Few-Shot {fs_id}: {fs_gold}")

def baue_messages_B(text: str) -> list[dict]:
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]   # Baseline-Prompt für faire Δ-Messung gegen A
    for fs_text, fs_gold in few_shots:
        msgs.append({"role": "user",      "content": f"Stellenanzeige:\n\n{fs_text}"})
        msgs.append({"role": "assistant", "content": json.dumps(fs_gold, ensure_ascii=False)})
    msgs.append({"role": "user", "content": f"Stellenanzeige:\n\n{text[:MAX_INPUT_CHARS_B]}"})
    return msgs

def inferenz_B(text: str) -> tuple[dict | None, str]:
    messages = baue_messages_B(text)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return extrahiere_json(response), response


# Run Iteration B
PRED_PATH_B = Path("predictions_iter_B.jsonl")
predictions_B, parse_fails_B = [], 0
t_start = time.time()

with PRED_PATH_B.open("w", encoding="utf-8") as f:
    for i, row in enumerate(anzeigen.itertuples(index=False), 1):
        t_i = time.time()
        parsed, raw = inferenz_B(row.text)
        dt = time.time() - t_i
        if parsed is None:
            parse_fails_B += 1
            entry, tag = {"refnr": row.refnr, "_parse_fail": True, "_raw": raw[:400]}, "PARSE_FAIL"
        else:
            entry, tag = {"refnr": row.refnr, **parsed}, "OK"
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")
        predictions_B.append(entry)
        print(f"  [{i:02d}/{len(anzeigen)}] {row.refnr}  {tag:<10}  ({dt:.1f}s)")

print(f"\nIteration B fertig: {len(predictions_B)} in {time.time()-t_start:.0f}s, {parse_fails_B} Parse-Fails")
print(f"Predictions → {PRED_PATH_B.resolve()}")

## Phase 6 — Voller 7B-Run (gauss)

Beste Pipeline aus Phase 4 (Iteration A oder B — wähle die mit höherer Gesamt-Accuracy in `03_eval.ipynb`) auf den **vollen Korpus** (90 Anzeigen) loslassen.

- Output: `predictions_7b_full.jsonl`
- Erwartete Laufzeit: ~5–8 min (90 × ~4 s)
- Im Eval-Notebook prüfen: Accuracy auf den 12 Gold bleibt stabil + Schema-Konformität auf den restlichen 78 via `validate.py --validate-jsonl`

Setze `FINAL_INFERENZ` auf die Funktion deiner besseren Iteration (`inferenz_A` oder `inferenz_B`).

In [ ]:
# Beste Pipeline aus Phase 4 wählen — Default: Iteration A (Prompt-Klarstellung)
# Falls Iteration B besser war: FINAL_INFERENZ = inferenz_B umstellen
FINAL_INFERENZ = inferenz_A
PRED_PATH_FULL = Path("predictions_7b_full.jsonl")

# Voller Korpus (alle 90 Anzeigen, nicht nur die 12 Gold)
korpus_full = pd.read_json(KORPUS_PATH, lines=True)
print(f"Voller Korpus: {len(korpus_full)} Anzeigen")

predictions_full, parse_fails_full = [], 0
t_start = time.time()

with PRED_PATH_FULL.open("w", encoding="utf-8") as f:
    for i, row in enumerate(korpus_full.itertuples(index=False), 1):
        t_i = time.time()
        parsed, raw = FINAL_INFERENZ(row.text)
        dt = time.time() - t_i
        if parsed is None:
            parse_fails_full += 1
            entry, tag = {"refnr": row.refnr, "_parse_fail": True, "_raw": raw[:400]}, "PARSE_FAIL"
        else:
            entry, tag = {"refnr": row.refnr, **parsed}, "OK"
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")
        predictions_full.append(entry)
        if i % 10 == 0 or tag == "PARSE_FAIL":
            print(f"  [{i:03d}/{len(korpus_full)}] {row.refnr}  {tag:<10}  ({dt:.1f}s)")

elapsed = time.time() - t_start
print(f"\nVoller 7B-Run fertig: {len(predictions_full)} Anzeigen in {elapsed:.0f}s "
      f"({elapsed/len(predictions_full):.1f}s/Anzeige), {parse_fails_full} Parse-Fails")
print(f"Predictions → {PRED_PATH_FULL.resolve()}")
print("\nDanach im Terminal: python annotation/validate.py --validate-jsonl predictions_7b_full.jsonl")

## Phase 6 — 3B-Run (euler)

**Server-Wechsel:** Spawn auf **euler** neu starten (Home-Verzeichnis ist NFS-synchronisiert, deine Dateien sind drüben da). Auf euler hat die V100 nur 16 GB — 7B passt nicht rein, 3B (~12 GB fp32) läuft komfortabel.

- Modell: `Qwen/Qwen2.5-3B-Instruct`
- Output: `predictions_3b_full.jsonl`
- Kernel-Restart auf euler nötig — Imports + Modell-Load erneut ausführen (die Zellen oben funktionieren 1:1, nur `MODEL_NAME` umstellen)

Diese Zelle setzt das nötige um. Wichtig: pipeline-Logik (`FINAL_INFERENZ`) bleibt identisch zur 7B-Variante — der einzige Unterschied ist die Modell-Größe.

In [ ]:
# Auf euler nach Kernel-Restart: erst die Imports + Setup-Zellen oben neu laufen lassen,
# DANN MODEL_NAME überschreiben und neu laden:

MODEL_NAME_3B = "Qwen/Qwen2.5-3B-Instruct"

# Cleanup alter 7B-Speicher (falls noch im Notebook-Kernel)
import gc
try:
    del model, tokenizer
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

# Modell + Tokenizer neu (3B) — wieder fp32 + .to("cuda").eval()
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_3B)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_3B,
    torch_dtype=torch.float32,
).to("cuda").eval()
print(f"Modell {MODEL_NAME_3B} geladen in {time.time()-t0:.1f}s")
print(f"GPU-Speicher belegt: {torch.cuda.memory_allocated() / 1e9:.1f} GB / "
      f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Run 3B auf vollem Korpus mit derselben FINAL_INFERENZ-Funktion (verweist auf model/tokenizer → automatisch 3B)
PRED_PATH_3B = Path("predictions_3b_full.jsonl")
korpus_full = pd.read_json(KORPUS_PATH, lines=True)

predictions_3b, parse_fails_3b = [], 0
t_start = time.time()

with PRED_PATH_3B.open("w", encoding="utf-8") as f:
    for i, row in enumerate(korpus_full.itertuples(index=False), 1):
        t_i = time.time()
        parsed, raw = FINAL_INFERENZ(row.text)
        dt = time.time() - t_i
        if parsed is None:
            parse_fails_3b += 1
            entry, tag = {"refnr": row.refnr, "_parse_fail": True, "_raw": raw[:400]}, "PARSE_FAIL"
        else:
            entry, tag = {"refnr": row.refnr, **parsed}, "OK"
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")
        predictions_3b.append(entry)
        if i % 10 == 0 or tag == "PARSE_FAIL":
            print(f"  [{i:03d}/{len(korpus_full)}] {row.refnr}  {tag:<10}  ({dt:.1f}s)")

elapsed = time.time() - t_start
print(f"\n3B-Run fertig: {len(predictions_3b)} Anzeigen in {elapsed:.0f}s "
      f"({elapsed/len(predictions_3b):.1f}s/Anzeige), {parse_fails_3b} Parse-Fails")
print(f"Predictions → {PRED_PATH_3B.resolve()}")